# Parabolic Drop Reversal (RSI < 10) on SPY
## Strategy Brief
The Parabolic Drop Reversal strategy aims to identify extreme oversold conditions in SPY using the Relative Strength Index (RSI). When the RSI drops below 10, it signals a potential reversal, predicting a mean reversion in the price. The trade logic involves buying SPY when the RSI is less than 10 and selling when the RSI moves back above 30. Historical testing suggests that this strategy can capture short-term rebounds, but it may underperform in trending markets.
## References
- (No external references)

## PHASE 1 - Trading Context
In this phase, we define the parameters and constants required for our strategy, including the RSI thresholds for entry and exit, and the lookback period for RSI calculation.

In [ ]:
RSI_PERIOD = 14
RSI_ENTRY_THRESHOLD = 10
RSI_EXIT_THRESHOLD = 30
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
TICKER = 'SPY'

## PHASE 2 - Data Exploration
We will download historical price data for SPY from Yahoo Finance, calculate the RSI, and plot it alongside the price data to visualize potential entry and exit points.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Calculate RSI
delta = data['Adj Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=RSI_PERIOD).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=RSI_PERIOD).mean()
rs = gain / loss
rsi = 100 - (100 / (1 + rs))
data['RSI'] = rsi

# Plot
plt.figure(figsize=(14, 7))
plt.subplot(2, 1, 1)
plt.plot(data['Adj Close'], label='SPY Price')
plt.title('SPY Price and RSI')
plt.legend()
plt.subplot(2, 1, 2)
plt.plot(data['RSI'], label='RSI', color='orange')
plt.axhline(RSI_ENTRY_THRESHOLD, color='red', linestyle='--', label='RSI Entry Threshold')
plt.axhline(RSI_EXIT_THRESHOLD, color='green', linestyle='--', label='RSI Exit Threshold')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We will create a signal series based on the RSI values and define the entry and exit logic. The positions series will indicate when to be long SPY.

In [ ]:
# Create signal series
data['Signal'] = 0

# Entry logic
entry_condition = (data['RSI'] < RSI_ENTRY_THRESHOLD)
data.loc[entry_condition, 'Signal'] = 1

# Exit logic
exit_condition = (data['RSI'] > RSI_EXIT_THRESHOLD)
data.loc[exit_condition, 'Signal'] = 0

# Forward fill positions
data['Position'] = data['Signal'].ffill().fillna(0)

## PHASE 4 - Coding & Backtesting
We will shift the positions by one day to simulate trading at the close of the signal day, calculate daily returns, and plot the equity curve.

In [ ]:
# Shift positions for backtesting
positions = data['Position'].shift(1)

# Calculate daily returns
data['Market Return'] = data['Adj Close'].pct_change()
data['Strategy Return'] = positions * data['Market Return']

# Calculate equity curve
data['Equity Curve'] = (1 + data['Strategy Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity Curve'], label='Strategy Equity Curve')
plt.title('Equity Curve')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We will calculate key performance metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. A comparison table against a buy-and-hold strategy will be provided.

In [ ]:
def calculate_performance_metrics(data):
    # Calculate CAGR
    total_return = data['Equity Curve'].iloc[-1] - 1
    num_years = (data.index[-1] - data.index[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / num_years) - 1

    # Calculate Sharpe ratio
    sharpe_ratio = data['Strategy Return'].mean() / data['Strategy Return'].std() * np.sqrt(252)

    # Calculate Sortino ratio
    downside_std = data[data['Strategy Return'] < 0]['Strategy Return'].std()
    sortino_ratio = data['Strategy Return'].mean() / downside_std * np.sqrt(252)

    # Calculate Calmar ratio
    max_drawdown = (data['Equity Curve'].cummax() - data['Equity Curve']).max()
    calmar_ratio = cagr / max_drawdown

    # Buy and hold strategy
    buy_and_hold_return = data['Adj Close'].iloc[-1] / data['Adj Close'].iloc[0] - 1
    buy_and_hold_cagr = (1 + buy_and_hold_return) ** (1 / num_years) - 1

    # Print metrics
    print(f"CAGR: {cagr:.2%}")
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"Sortino Ratio: {sortino_ratio:.2f}")
    print(f"Calmar Ratio: {calmar_ratio:.2f}")
    print(f"Max Drawdown: {max_drawdown:.2%}")
    print(f"Buy and Hold CAGR: {buy_and_hold_cagr:.2%}")

calculate_performance_metrics(data)

## PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of SPY data, compute today's RSI signal, and print the suggested position.

In [ ]:
def get_latest_signal():
    # Download last 60 days of data
    recent_data = yf.download(TICKER, period='60d')
    
    # Calculate RSI
    delta = recent_data['Adj Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=RSI_PERIOD).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=RSI_PERIOD).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    recent_data['RSI'] = rsi
    
    # Determine position
    latest_rsi = recent_data['RSI'].iloc[-1]
    if latest_rsi < RSI_ENTRY_THRESHOLD:
        position = 1  # Buy
    elif latest_rsi > RSI_EXIT_THRESHOLD:
        position = 0  # Sell
    else:
        position = np.nan  # Hold
    
    print(f"Latest RSI: {latest_rsi:.2f}, Suggested Position: {position}")

get_latest_signal()